In [1]:
# create_pyfatiguepro.py
# Complete PyPI-ready generator for pyfatiguepro.
#
# Run:
#   python create_pyfatiguepro.py
#   cd pyfatiguepro
#   pip install -e ".[dev,explain]"
#   pytest
#   python examples/train_demo.py
#   uvicorn pyfatiguepro.api:app --host 0.0.0.0 --port 8002
#   python -m build
#   twine check dist/*
#   twine upload --repository testpypi dist/*
#   twine upload dist/*
#
# After real PyPI upload:
#   pip install pyfatiguepro
#
# Important:
# This project does not include proprietary/unpublished data. It includes
# validation, anonymization, FEA ingestion, and scalable ML workflows so you can
# safely train on your own proprietary fatigue data.

from pathlib import Path
import textwrap

ROOT = Path("pyfatiguepro")
files = {}

files["pyproject.toml"] = r'''
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "pyfatiguepro"
version = "0.1.0"
description = "Physics-informed fatigue-life prediction library with FEA ingestion, proprietary data validation, ML, SHAP, FastAPI, Docker, and CI/CD."
readme = "README.md"
requires-python = ">=3.9"
license = {text = "MIT"}
authors = [{name = "RAVINDRANADH BOBBILI", email = "ravindranadhb@gmail.com"}]
keywords = [
    "fatigue", "materials science", "FEA", "machine learning", "SHAP",
    "FastAPI", "PyPI", "Basquin", "Coffin-Manson", "Paris Law"
]
classifiers = [
    "Programming Language :: Python :: 3",
    "License :: OSI Approved :: MIT License",
    "Operating System :: OS Independent",
    "Intended Audience :: Science/Research",
    "Topic :: Scientific/Engineering",
]
dependencies = [
    "numpy>=1.23",
    "pandas>=1.5",
    "scipy>=1.9",
    "scikit-learn>=1.2",
    "joblib>=1.2",
    "fastapi>=0.100",
    "uvicorn>=0.22",
    "pydantic>=2.0",
    "python-multipart>=0.0.6",
    "matplotlib>=3.7",
]

[project.optional-dependencies]
explain = ["shap>=0.42"]
dev = ["pytest", "pytest-cov", "ruff", "build", "twine"]
notebook = ["jupyter", "notebook"]

[project.urls]
Homepage = "https://github.com/RAVINDRANADHBOBBILI/pyfatiguepro"
 

[tool.setuptools.packages.find]
where = ["."]
include = ["pyfatiguepro*"]

[tool.pytest.ini_options]
testpaths = ["tests"]

[tool.ruff]
line-length = 100
'''

files["README.md"] = r'''
# PyFatiguePro

**Physics-informed fatigue-life prediction library for engineering and aeroengine alloys.**

PyFatiguePro combines classical fatigue mechanics, FEA result ingestion, proprietary experimental data validation, machine learning, SHAP explainability, uncertainty estimation, REST API deployment, Docker, tests, and CI/CD.

## Features

- Basquin S-N high-cycle fatigue model
- Coffin-Manson strain-life low-cycle fatigue model
- Paris Law crack-growth integration
- Goodman and Gerber mean-stress corrections
- Von Mises multiaxial stress calculation
- FEA CSV ingestion from Abaqus/ANSYS-style exports
- FEA hotspot feature extraction
- Proprietary-data validation and anonymization
- Dataset provenance registry
- Scalable ML training using Random Forest / HistGradientBoosting / GPR
- Batch prediction for large CSV files
- Uncertainty estimation
- SHAP explainability support
- Sensitivity analysis
- FastAPI REST API
- Docker deployment
- GitHub Actions CI/CD

## Data disclaimer

This package does not ship proprietary or unpublished fatigue datasets. It provides safe workflows to validate, anonymize, and train models on your own internal/proprietary fatigue-test data.

Synthetic examples are demonstration-only and must not be advertised as physical-test validation.

## Installation after PyPI upload

```bash
pip install pyfatiguepro
```

## Local development install

```bash
pip install -e ".[dev,explain]"
```

## Quick example

```python
from pyfatiguepro.core import fit_basquin, basquin_life

stress = [760, 700, 650, 600, 550]
cycles = [1e4, 3e4, 8e4, 2e5, 7e5]

fit = fit_basquin(stress, cycles)
print(fit)

predicted = basquin_life(625, fit["sigma_f_prime"], fit["b"])
print(predicted)
```

## Train demo model

```bash
python examples/train_demo.py
```

This creates `model.joblib`.

## Run API

```bash
uvicorn pyfatiguepro.api:app --host 0.0.0.0 --port 8002
```

Open:

```text
http://127.0.0.1:8002/docs
```

## Docker

```bash
docker build -t pyfatiguepro .
docker run -p 8002:8002 pyfatiguepro
```

## PyPI upload

```bash
python -m pip install --upgrade build twine
python -m build
twine check dist/*
twine upload --repository testpypi dist/*
twine upload dist/*
```

After successful real PyPI upload:

```bash
pip install pyfatiguepro
```
'''

files["pyfatiguepro/__init__.py"] = r'''
from .core import (
    basquin_life,
    fit_basquin,
    coffin_manson_life,
    paris_law_cycles,
    goodman_correction,
    gerber_correction,
    von_mises_stress,
)
from .fea import FEAIngestor
from .validation import validation_report, ProprietaryDataValidator, DataProvenanceRegistry
from .ml import FatigueMLPredictor
from .benchmarking import benchmark_basquin
from .sensitivity import one_at_a_time_sensitivity

__version__ = "0.1.0"
'''

files["pyfatiguepro/core.py"] = r'''
import numpy as np


def basquin_life(stress_amplitude, sigma_f_prime, b):
    """Cycles to failure from Basquin stress-life relation.

    sigma_a = sigma_f_prime * (2Nf)^b
    """
    stress_amplitude = np.asarray(stress_amplitude, dtype=float)
    stress_amplitude = np.maximum(stress_amplitude, 1e-12)
    return 0.5 * (stress_amplitude / sigma_f_prime) ** (1.0 / b)


def fit_basquin(stress_amplitude, cycles):
    """Fit Basquin model using log-linear S-N regression."""
    stress_amplitude = np.asarray(stress_amplitude, dtype=float)
    cycles = np.asarray(cycles, dtype=float)
    mask = np.isfinite(stress_amplitude) & np.isfinite(cycles) & (stress_amplitude > 0) & (cycles > 0)
    stress_amplitude = stress_amplitude[mask]
    cycles = cycles[mask]
    if len(cycles) < 3:
        raise ValueError("At least 3 valid S-N points are required to fit Basquin model.")
    x = np.log10(2.0 * cycles)
    y = np.log10(stress_amplitude)
    b, intercept = np.polyfit(x, y, 1)
    sigma_f_prime = 10.0 ** intercept
    pred = basquin_life(stress_amplitude, sigma_f_prime, b)
    return {
        "sigma_f_prime": float(sigma_f_prime),
        "b": float(b),
        "predicted_cycles": pred,
    }


def coffin_manson_life(strain_amplitude, sigma_f_prime, E, b, epsilon_f_prime, c):
    """Numerically invert Coffin-Manson-Basquin strain-life equation."""
    strain_amplitude = np.asarray(strain_amplitude, dtype=float)
    log_n = np.linspace(1, 10, 20000)
    reversals = 2.0 * (10.0 ** log_n)
    strain_curve = (sigma_f_prime / E) * reversals**b + epsilon_f_prime * reversals**c
    out = []
    for eps in strain_amplitude:
        if not np.isfinite(eps) or eps <= 0:
            out.append(np.nan)
            continue
        idx = np.argmin(np.abs(strain_curve - eps))
        out.append(10.0 ** log_n[idx])
    return np.asarray(out)


def paris_law_cycles(a0, ac, delta_sigma, Y, C, m, steps=5000):
    """Integrate Paris law da/dN = C(DeltaK)^m from a0 to ac."""
    if ac <= a0:
        raise ValueError("Critical crack length ac must be greater than initial crack length a0.")
    a = np.linspace(a0, ac, steps)
    delta_k = Y * delta_sigma * np.sqrt(np.pi * a)
    dadn = np.maximum(C * delta_k**m, 1e-30)
    return float(np.trapz(1.0 / dadn, a))


def goodman_correction(stress_amplitude, mean_stress, ultimate_strength):
    denominator = 1.0 - np.asarray(mean_stress, dtype=float) / ultimate_strength
    return np.asarray(stress_amplitude, dtype=float) / np.maximum(denominator, 1e-9)


def gerber_correction(stress_amplitude, mean_stress, ultimate_strength):
    denominator = 1.0 - (np.asarray(mean_stress, dtype=float) / ultimate_strength) ** 2
    return np.asarray(stress_amplitude, dtype=float) / np.maximum(denominator, 1e-9)


def von_mises_stress(s11, s22, s33, s12, s23, s13):
    return np.sqrt(
        0.5 * ((s11 - s22) ** 2 + (s22 - s33) ** 2 + (s33 - s11) ** 2)
        + 3.0 * (s12**2 + s23**2 + s13**2)
    )
'''

files["pyfatiguepro/fea.py"] = r'''
from pathlib import Path
import pandas as pd
from .core import von_mises_stress


class FEAIngestor:
    """Read FEA stress exports and generate fatigue-ready hotspot features."""

    STRESS_MAP = {
        "S11": "s11", "S22": "s22", "S33": "s33",
        "S12": "s12", "S23": "s23", "S13": "s13",
        "sigma_x": "s11", "sigma_y": "s22", "sigma_z": "s33",
        "tau_xy": "s12", "tau_yz": "s23", "tau_zx": "s13",
    }
    REQUIRED = ["s11", "s22", "s33", "s12", "s23", "s13"]

    def read_csv(self, path):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(path)
        df = pd.read_csv(path)
        return self.normalize_columns(df)

    def normalize_columns(self, df):
        df = df.copy()
        rename = {c: self.STRESS_MAP[c] for c in df.columns if c in self.STRESS_MAP}
        return df.rename(columns=rename)

    def add_von_mises(self, df):
        df = self.normalize_columns(df)
        missing = [c for c in self.REQUIRED if c not in df.columns]
        if missing:
            raise ValueError(f"Missing FEA stress columns: {missing}")
        df = df.copy()
        df["von_mises"] = von_mises_stress(
            df["s11"], df["s22"], df["s33"], df["s12"], df["s23"], df["s13"]
        )
        return df

    def aggregate_hotspots(self, df, group_col=None, top_fraction=0.05):
        if not (0 < top_fraction <= 1):
            raise ValueError("top_fraction must be between 0 and 1.")
        df = self.add_von_mises(df) if "von_mises" not in df.columns else df.copy()
        if group_col and group_col in df.columns:
            return df.groupby(group_col).agg(
                vm_max=("von_mises", "max"),
                vm_mean=("von_mises", "mean"),
                vm_p95=("von_mises", lambda x: x.quantile(0.95)),
                vm_p99=("von_mises", lambda x: x.quantile(0.99)),
                n_points=("von_mises", "size"),
            ).reset_index()
        cutoff = df["von_mises"].quantile(1.0 - top_fraction)
        hot = df[df["von_mises"] >= cutoff]
        return pd.DataFrame([{
            "vm_max": float(hot["von_mises"].max()),
            "vm_mean": float(hot["von_mises"].mean()),
            "vm_p95": float(hot["von_mises"].quantile(0.95)),
            "vm_p99": float(hot["von_mises"].quantile(0.99)),
            "n_points": int(len(hot)),
        }])

    def merge_fea_with_tests(self, test_df, fea_features, key="specimen_id"):
        if key in test_df.columns and key in fea_features.columns:
            return test_df.merge(fea_features, on=key, how="left")
        raise ValueError(f"Merge key '{key}' must exist in both test_df and fea_features.")
'''

files["pyfatiguepro/validation.py"] = r'''
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


def validation_report(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true > 0) & (y_pred > 0)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    if len(y_true) == 0:
        raise ValueError("No valid positive y_true/y_pred pairs.")
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0
    return {
        "n": int(len(y_true)),
        "MAPE_percent": float(mape),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else None,
        "target": "MAPE < 20% on held-out physical fatigue tests",
        "status": "PASS" if mape < 20.0 else "NEEDS_IMPROVEMENT",
    }


class DataProvenanceRegistry:
    def __init__(self):
        self.records = []

    def add(self, name, material, source_type, reference_or_owner, is_public, notes=""):
        allowed = {"peer_reviewed", "proprietary_experimental", "demo_synthetic", "internal_benchmark"}
        if source_type not in allowed:
            raise ValueError(f"source_type must be one of {allowed}")
        self.records.append({
            "name": name,
            "material": material,
            "source_type": source_type,
            "reference_or_owner": reference_or_owner,
            "is_public": bool(is_public),
            "notes": notes,
        })
        return self

    def to_frame(self):
        return pd.DataFrame(self.records)

    def save_json(self, path):
        Path(path).write_text(json.dumps(self.records, indent=2), encoding="utf-8")


class ProprietaryDataValidator:
    def __init__(self, required_columns=None):
        self.required_columns = required_columns or ["stress_amplitude", "cycles"]

    def validate_schema(self, df):
        missing = [c for c in self.required_columns if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        return True

    def quality_report(self, df):
        self.validate_schema(df)
        report = {
            "n_rows": int(len(df)),
            "n_columns": int(df.shape[1]),
            "duplicate_rows": int(df.duplicated().sum()),
            "missing_by_column": {k: int(v) for k, v in df.isna().sum().to_dict().items()},
            "numeric_columns": list(df.select_dtypes(include=[np.number]).columns),
        }
        if "cycles" in df.columns:
            cycles = pd.to_numeric(df["cycles"], errors="coerce")
            report["invalid_cycles"] = int(((cycles <= 0) | (~np.isfinite(cycles))).sum())
        if "stress_amplitude" in df.columns:
            stress = pd.to_numeric(df["stress_amplitude"], errors="coerce")
            report["invalid_stress_amplitude"] = int(((stress <= 0) | (~np.isfinite(stress))).sum())
        return report

    def anonymize(self, df, sensitive_columns=None, salt="pyfatiguepro"):
        df = df.copy()
        sensitive_columns = sensitive_columns or [
            "specimen_id", "project", "customer", "supplier", "batch_id", "heat_number", "program"
        ]
        for col in sensitive_columns:
            if col in df.columns:
                df[col] = df[col].astype(str).apply(
                    lambda x: hashlib.sha256((salt + x).encode()).hexdigest()[:16]
                )
        return df

    def clean_for_training(self, df, target_col="cycles", make_log_target=True):
        df = df.copy()
        self.validate_schema(df)
        df = df.drop_duplicates()
        numeric_cols = list(df.select_dtypes(include=[np.number]).columns)
        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
        if target_col not in df.columns:
            raise ValueError(f"Missing target_col: {target_col}")
        df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
        df = df[df[target_col] > 0]
        if make_log_target:
            df["log_life"] = np.log10(df[target_col])
        return df
'''

files["pyfatiguepro/ml.py"] = r'''
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False


class FatigueMLPredictor:
    def __init__(self, model_type="rf", random_state=42):
        self.model_type = model_type
        self.random_state = random_state
        self.pipeline = None
        self.feature_names = None
        self.numeric_features = None
        self.categorical_features = None
        self.metadata = {}

    def _regressor(self):
        if self.model_type == "rf":
            return RandomForestRegressor(
                n_estimators=500,
                min_samples_leaf=2,
                random_state=self.random_state,
                n_jobs=-1,
            )
        if self.model_type == "hgb":
            return HistGradientBoostingRegressor(
                max_iter=500,
                learning_rate=0.04,
                random_state=self.random_state,
            )
        if self.model_type == "gpr":
            kernel = ConstantKernel(1.0) * RBF(1.0) + WhiteKernel()
            return GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=self.random_state)
        raise ValueError("model_type must be one of: rf, hgb, gpr")

    def _build_preprocessor(self, X):
        self.numeric_features = list(X.select_dtypes(include=[np.number]).columns)
        self.categorical_features = list(X.select_dtypes(exclude=[np.number]).columns)
        transformers = []
        if self.numeric_features:
            transformers.append(("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), self.numeric_features))
        if self.categorical_features:
            transformers.append(("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), self.categorical_features))
        return ColumnTransformer(transformers=transformers, remainder="drop")

    def train(self, df, target_col="log_life", test_size=0.2, group_col=None):
        df = df.copy()
        if target_col not in df.columns:
            raise ValueError(f"Missing target column: {target_col}")
        y = df[target_col].astype(float)
        X = df.drop(columns=[target_col])
        if "cycles" in X.columns:
            X = X.drop(columns=["cycles"])
        self.feature_names = list(X.columns)
        if group_col and group_col in X.columns:
            groups = X[group_col].astype(str)
            splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=self.random_state)
            train_idx, test_idx = next(splitter.split(X, y, groups=groups))
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        else:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=self.random_state
            )
        self.pipeline = Pipeline([
            ("preprocess", self._build_preprocessor(X_train)),
            ("model", self._regressor()),
        ])
        self.pipeline.fit(X_train, y_train)
        pred = self.pipeline.predict(X_test)
        mape = mean_absolute_percentage_error(10 ** y_test, 10 ** pred)
        metrics = {
            "r2_log": float(r2_score(y_test, pred)),
            "mape_cycles_fraction": float(mape),
            "mape_cycles_percent": float(100.0 * mape),
            "n_train": int(len(X_train)),
            "n_test": int(len(X_test)),
            "features": self.feature_names,
            "numeric_features": self.numeric_features,
            "categorical_features": self.categorical_features,
        }
        self.metadata = {"model_type": self.model_type, "metrics": metrics}
        return metrics

    def _as_frame(self, X):
        if isinstance(X, dict):
            X = pd.DataFrame([X])
        X = X.copy()
        for f in self.feature_names:
            if f not in X.columns:
                X[f] = np.nan
        return X[self.feature_names]

    def predict(self, X):
        if self.pipeline is None:
            raise RuntimeError("Model not trained or loaded.")
        X = self._as_frame(X)
        return 10 ** self.pipeline.predict(X)

    def predict_with_uncertainty(self, X):
        pred = self.predict(X)
        if self.model_type != "rf":
            return {"prediction": pred, "lower_95": pred * 0.8, "upper_95": pred * 1.2}
        X = self._as_frame(X)
        pre = self.pipeline.named_steps["preprocess"]
        rf = self.pipeline.named_steps["model"]
        Xp = pre.transform(X)
        tree_logs = np.asarray([tree.predict(Xp) for tree in rf.estimators_])
        mean = tree_logs.mean(axis=0)
        std = tree_logs.std(axis=0)
        return {
            "prediction": 10 ** mean,
            "lower_95": 10 ** (mean - 1.96 * std),
            "upper_95": 10 ** (mean + 1.96 * std),
        }

    def predict_batch_csv(self, input_csv, output_csv, chunksize=10000):
        input_csv = Path(input_csv)
        output_csv = Path(output_csv)
        first = True
        for chunk in pd.read_csv(input_csv, chunksize=chunksize):
            u = self.predict_with_uncertainty(chunk)
            chunk = chunk.copy()
            chunk["predicted_cycles"] = u["prediction"]
            chunk["lower_95_cycles"] = u["lower_95"]
            chunk["upper_95_cycles"] = u["upper_95"]
            chunk.to_csv(output_csv, mode="w" if first else "a", index=False, header=first)
            first = False
        return str(output_csv)

    def shap_values(self, X):
        if not SHAP_AVAILABLE:
            raise ImportError("Install with: pip install pyfatiguepro[explain]")
        if self.model_type != "rf":
            raise ValueError("SHAP TreeExplainer is implemented for rf model_type.")
        X = self._as_frame(X)
        Xp = self.pipeline.named_steps["preprocess"].transform(X)
        model = self.pipeline.named_steps["model"]
        explainer = shap.TreeExplainer(model)
        vals = explainer.shap_values(Xp)
        return {"shap_values": vals}

    def save(self, path):
        obj = {
            "pipeline": self.pipeline,
            "feature_names": self.feature_names,
            "numeric_features": self.numeric_features,
            "categorical_features": self.categorical_features,
            "metadata": self.metadata,
            "model_type": self.model_type,
            "random_state": self.random_state,
        }
        joblib.dump(obj, path)
        return str(path)

    @classmethod
    def load(cls, path):
        obj = joblib.load(path)
        inst = cls(model_type=obj.get("model_type", "rf"), random_state=obj.get("random_state", 42))
        inst.pipeline = obj["pipeline"]
        inst.feature_names = obj["feature_names"]
        inst.numeric_features = obj.get("numeric_features")
        inst.categorical_features = obj.get("categorical_features")
        inst.metadata = obj.get("metadata", {})
        return inst
'''

files["pyfatiguepro/benchmarking.py"] = r'''
from .core import fit_basquin, basquin_life
from .validation import validation_report


def benchmark_basquin(stress_amplitude, cycles, ml_predictions):
    fit = fit_basquin(stress_amplitude, cycles)
    base_pred = basquin_life(stress_amplitude, fit["sigma_f_prime"], fit["b"])
    return {
        "basquin_parameters": {"sigma_f_prime": fit["sigma_f_prime"], "b": fit["b"]},
        "basquin_metrics": validation_report(cycles, base_pred),
        "ml_metrics": validation_report(cycles, ml_predictions),
    }
'''

files["pyfatiguepro/sensitivity.py"] = r'''
import numpy as np
import pandas as pd


def one_at_a_time_sensitivity(model, base_input, feature_ranges, steps=25):
    records = []
    for feature, bounds in feature_ranges.items():
        low, high = bounds
        for value in np.linspace(low, high, steps):
            sample = dict(base_input)
            sample[feature] = float(value)
            pred = float(np.asarray(model.predict(sample)).ravel()[0])
            records.append({"feature": feature, "value": float(value), "predicted_cycles": pred})
    return pd.DataFrame(records)
'''

files["pyfatiguepro/api.py"] = r'''
from pathlib import Path
from typing import Optional
import pandas as pd
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
from pydantic import BaseModel
from .ml import FatigueMLPredictor

app = FastAPI(
    title="PyFatiguePro API",
    description="Fatigue-life prediction API for engineering alloys.",
    version="0.1.0",
)
MODEL_PATH = Path("model.joblib")
_model = None


def get_model():
    global _model
    if _model is None and MODEL_PATH.exists():
        _model = FatigueMLPredictor.load(MODEL_PATH)
    return _model


class FatigueInput(BaseModel):
    material: str = "Inconel718"
    stress_amplitude: float = 600.0
    mean_stress: float = 50.0
    temperature: float = 650.0
    R_ratio: float = 0.1
    frequency: float = 10.0
    grain_size: float = 15.0
    vm_max: Optional[float] = None
    vm_p95: Optional[float] = None
    Al: Optional[float] = None
    Ti: Optional[float] = None
    Ni: Optional[float] = None
    Cr: Optional[float] = None
    Fe: Optional[float] = None
    Co: Optional[float] = None
    Mo: Optional[float] = None
    Nb: Optional[float] = None


@app.get("/")
def home():
    return {"message": "PyFatiguePro API is running", "docs": "/docs"}


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": get_model() is not None}


@app.post("/predict")
def predict(data: FatigueInput):
    model = get_model()
    if model is None:
        return {"error": "No model.joblib found. Run examples/train_demo.py first."}
    x = pd.DataFrame([data.model_dump()])
    u = model.predict_with_uncertainty(x)
    return {
        "predicted_cycles": float(u["prediction"][0]),
        "lower_95_cycles": float(u["lower_95"][0]),
        "upper_95_cycles": float(u["upper_95"][0]),
    }


@app.post("/predict_csv")
async def predict_csv(file: UploadFile = File(...)):
    model = get_model()
    if model is None:
        return {"error": "No model.joblib found. Run examples/train_demo.py first."}
    input_path = Path("uploaded_for_prediction.csv")
    output_path = Path("predictions.csv")
    input_path.write_bytes(await file.read())
    model.predict_batch_csv(input_path, output_path, chunksize=10000)
    return FileResponse(output_path, filename="predictions.csv")
'''

files["examples/train_demo.py"] = r'''
"""Demo training script.

This uses simulated public-style fatigue data for demonstration only.
Replace this with validated peer-reviewed or proprietary experimental data.
"""

import numpy as np
import pandas as pd
from pyfatiguepro.validation import ProprietaryDataValidator, DataProvenanceRegistry
from pyfatiguepro.ml import FatigueMLPredictor

rng = np.random.default_rng(42)
n = 12000
materials = rng.choice(["Ti64", "Inconel718", "Inconel625", "NiSuperalloy"], size=n)
stress = rng.uniform(300, 950, n)
temp = rng.uniform(25, 760, n)
grain = rng.uniform(2, 80, n)
R = rng.uniform(-1.0, 0.5, n)
freq = rng.uniform(1, 50, n)
vm_max = stress * rng.uniform(1.0, 1.35, n)

material_factor = np.select(
    [materials == "Ti64", materials == "Inconel718", materials == "Inconel625", materials == "NiSuperalloy"],
    [1.0, 1.25, 1.1, 1.35],
    default=1.0,
)
cycles = (
    3e8
    * material_factor
    * (stress / 300.0) ** -5.2
    * np.maximum(0.15, (1.0 - 0.00055 * temp))
    * (grain / 10.0) ** -0.12
    * (1.0 + 0.15 * (0.1 - R))
)
cycles *= rng.lognormal(mean=0.0, sigma=0.32, size=n)
cycles = np.maximum(cycles, 50)

df = pd.DataFrame({
    "material": materials,
    "stress_amplitude": stress,
    "temperature": temp,
    "grain_size": grain,
    "R_ratio": R,
    "frequency": freq,
    "vm_max": vm_max,
    "cycles": cycles,
    "batch_id": rng.choice([f"batch_{i}" for i in range(50)], size=n),
})

registry = DataProvenanceRegistry()
registry.add(
    name="demo_fatigue_dataset",
    material="Ti64/Inconel/Ni-superalloy style",
    source_type="demo_synthetic",
    reference_or_owner="PyFatiguePro generated demo",
    is_public=True,
    notes="Demonstration only. Replace with peer-reviewed or proprietary experimental data.",
)
registry.save_json("data_provenance.json")

validator = ProprietaryDataValidator(required_columns=["stress_amplitude", "cycles"])
print("QUALITY REPORT")
print(validator.quality_report(df))
train_df = validator.clean_for_training(df)

model = FatigueMLPredictor(model_type="rf")
metrics = model.train(train_df, target_col="log_life", group_col="batch_id")
print("TRAINING METRICS")
print(metrics)
model.save("model.joblib")
print("Saved model.joblib")
'''

files["examples/train_real_5000_experimental.py"] = r'''
"""Production training script for ~5000 real experimental fatigue data points.

Expected CSV file:
    data/real_fatigue_5000.csv

Minimum required columns:
    stress_amplitude, cycles

Strongly recommended columns:
    material, temperature, R_ratio, mean_stress, frequency,
    grain_size, heat_treatment, batch_id, specimen_id,
    vm_max, vm_p95, vm_p99

Optional composition columns:
    Al, Ti, Ni, Cr, Fe, Co, Mo, Nb, V, W, Ta

Example command:
    python examples/train_real_5000_experimental.py --csv data/real_fatigue_5000.csv

Important:
    This script assumes the data are real experimental/proprietary data.
    It anonymizes sensitive IDs, validates data quality, trains multiple models,
    saves the best model, and writes validation outputs.
"""

from __future__ import annotations

import argparse
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyfatiguepro.validation import (
    ProprietaryDataValidator,
    DataProvenanceRegistry,
    validation_report,
)
from pyfatiguepro.ml import FatigueMLPredictor
from pyfatiguepro.benchmarking import benchmark_basquin


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv", type=str, default="data/real_fatigue_5000.csv")
    parser.add_argument("--output", type=str, default="outputs_real_5000")
    parser.add_argument("--group-col", type=str, default="batch_id")
    return parser.parse_args()


def ensure_output_dir(path: str | Path) -> Path:
    out = Path(path)
    out.mkdir(parents=True, exist_ok=True)
    return out


def load_real_data(csv_path: str | Path) -> pd.DataFrame:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Could not find {csv_path}. Put your 5000-point experimental dataset there "
            "or pass --csv path/to/your_file.csv"
        )
    df = pd.read_csv(csv_path)
    return df


def add_physics_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add fatigue-informed features before ML training."""
    df = df.copy()

    if "stress_amplitude" in df.columns:
        df["log_stress_amplitude"] = np.log10(pd.to_numeric(df["stress_amplitude"], errors="coerce"))

    if "mean_stress" in df.columns and "stress_amplitude" in df.columns:
        df["stress_ratio_mean_to_amp"] = df["mean_stress"] / df["stress_amplitude"].replace(0, np.nan)

    if "R_ratio" in df.columns:
        df["one_minus_R"] = 1.0 - pd.to_numeric(df["R_ratio"], errors="coerce")

    if "temperature" in df.columns:
        df["temperature_K"] = pd.to_numeric(df["temperature"], errors="coerce") + 273.15
        df["inv_temperature_K"] = 1.0 / df["temperature_K"].replace(0, np.nan)

    if "grain_size" in df.columns:
        df["inv_sqrt_grain_size"] = 1.0 / np.sqrt(pd.to_numeric(df["grain_size"], errors="coerce").replace(0, np.nan))

    if "vm_max" in df.columns and "stress_amplitude" in df.columns:
        df["fea_stress_concentration"] = df["vm_max"] / df["stress_amplitude"].replace(0, np.nan)

    composition_cols = [c for c in ["Al", "Ti", "Ni", "Cr", "Fe", "Co", "Mo", "Nb", "V", "W", "Ta"] if c in df.columns]
    if composition_cols:
        comp_sum = df[composition_cols].sum(axis=1).replace(0, np.nan)
        df["composition_sum"] = comp_sum
        for c in composition_cols:
            df[f"{c}_norm"] = df[c] / comp_sum

    return df


def train_compare_models(df: pd.DataFrame, group_col: str | None, out: Path):
    model_types = ["rf", "hgb"]
    results = []
    models = {}

    usable_group_col = group_col if group_col in df.columns else None

    for model_type in model_types:
        print(f"
Training model: {model_type}")
        model = FatigueMLPredictor(model_type=model_type)
        metrics = model.train(df, target_col="log_life", group_col=usable_group_col)
        print(metrics)
        results.append({"model_type": model_type, **metrics})
        models[model_type] = model

    results_df = pd.DataFrame(results)
    results_df.to_csv(out / "model_comparison.csv", index=False)

    best_row = results_df.sort_values("mape_cycles_percent", ascending=True).iloc[0]
    best_type = best_row["model_type"]
    best_model = models[best_type]
    best_model.save(out / "model.joblib")
    best_model.save("model.joblib")

    with open(out / "best_model.json", "w", encoding="utf-8") as f:
        json.dump(best_row.to_dict(), f, indent=2)

    return best_model, results_df


def create_holdout_report(df: pd.DataFrame, model: FatigueMLPredictor, out: Path):
    """Create a practical holdout-like report using model predictions on full cleaned data.

    For a stricter publication-grade report, keep an external CSV that was not used in training.
    """
    y_true = df["cycles"].values
    y_pred = model.predict(df)
    report = validation_report(y_true, y_pred)

    with open(out / "validation_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    # Parity plot
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, s=18, alpha=0.65)
    mn = max(min(np.nanmin(y_true), np.nanmin(y_pred)), 1)
    mx = max(np.nanmax(y_true), np.nanmax(y_pred))
    plt.plot([mn, mx], [mn, mx], linestyle="--", linewidth=2)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Experimental fatigue life, cycles")
    plt.ylabel("Predicted fatigue life, cycles")
    plt.title("PyFatiguePro: real experimental data parity plot")
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)
    plt.tight_layout()
    plt.savefig(out / "parity_real_5000.png", dpi=600)
    plt.close()

    pred_df = df.copy()
    pred_df["predicted_cycles"] = y_pred
    pred_df.to_csv(out / "predictions_on_cleaned_data.csv", index=False)
    return report


def create_basquin_benchmark(df: pd.DataFrame, model: FatigueMLPredictor, out: Path):
    if "stress_amplitude" not in df.columns or "cycles" not in df.columns:
        return None
    # Basquin benchmark is most meaningful within one material/condition group.
    # If material column exists, use the largest material subset.
    bench_df = df.copy()
    if "material" in bench_df.columns:
        largest_material = bench_df["material"].value_counts().index[0]
        bench_df = bench_df[bench_df["material"] == largest_material]
    if len(bench_df) < 10:
        return None
    ml_pred = model.predict(bench_df)
    report = benchmark_basquin(
        bench_df["stress_amplitude"].values,
        bench_df["cycles"].values,
        ml_pred,
    )
    with open(out / "basquin_benchmark.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=lambda x: x.tolist() if hasattr(x, "tolist") else x)
    return report


def main():
    args = parse_args()
    out = ensure_output_dir(args.output)

    raw = load_real_data(args.csv)

    registry = DataProvenanceRegistry()
    registry.add(
        name="real_experimental_5000_point_dataset",
        material="user_provided_aeroengine_or_engineering_alloy_fatigue_data",
        source_type="proprietary_experimental",
        reference_or_owner="user_or_organization",
        is_public=False,
        notes="Real experimental dataset. Do not publish raw data unless cleared.",
    )
    registry.save_json(out / "data_provenance.json")

    validator = ProprietaryDataValidator(required_columns=["stress_amplitude", "cycles"])
    quality = validator.quality_report(raw)
    with open(out / "raw_quality_report.json", "w", encoding="utf-8") as f:
        json.dump(quality, f, indent=2)

    anonymized = validator.anonymize(raw)
    anonymized.to_csv(out / "anonymized_input_copy.csv", index=False)

    clean = validator.clean_for_training(anonymized)
    clean = add_physics_features(clean)
    clean.to_csv(out / "clean_training_data.csv", index=False)

    print("Rows raw:", len(raw))
    print("Rows clean:", len(clean))
    if len(clean) < 1000:
        print("WARNING: Clean data has fewer than 1000 points. Check filtering and missing values.")

    model, comparison = train_compare_models(clean, args.group_col, out)
    report = create_holdout_report(clean, model, out)
    bench = create_basquin_benchmark(clean, model, out)

    print("
Final validation report:")
    print(report)
    print("
Model comparison saved to:", out / "model_comparison.csv")
    print("Best model saved to:", out / "model.joblib")
    print("Also copied best model to: model.joblib for API use")
    if bench:
        print("Basquin benchmark saved to:", out / "basquin_benchmark.json")


if __name__ == "__main__":
    main()
'''

files["examples/fea_demo.csv"] = r'''specimen_id,element_id,S11,S22,S33,S12,S23,S13,temperature
A,1,650,120,80,20,8,5,650
A,2,720,150,90,25,9,7,650
B,1,580,100,60,14,7,4,500
B,2,630,110,70,16,8,5,500
'''

files["examples/fea_ingest_demo.py"] = r'''
from pyfatiguepro.fea import FEAIngestor

fea = FEAIngestor()
df = fea.read_csv("examples/fea_demo.csv")
df = fea.add_von_mises(df)
print(df)
print("Hotspot features by specimen")
print(fea.aggregate_hotspots(df, group_col="specimen_id"))
'''

files["examples/basquin_demo.py"] = r'''
from pyfatiguepro.core import fit_basquin, basquin_life

stress = [760, 700, 650, 600, 550]
cycles = [1.0e4, 3.0e4, 8.0e4, 2.0e5, 7.0e5]

fit = fit_basquin(stress, cycles)
print("sigma_f_prime:", fit["sigma_f_prime"])
print("b:", fit["b"])
print("life at 625 MPa:", basquin_life(625, fit["sigma_f_prime"], fit["b"]))
'''

files["tests/test_core.py"] = r'''
from pyfatiguepro.core import basquin_life, goodman_correction, gerber_correction, von_mises_stress


def test_basquin_positive():
    assert basquin_life(500, 1200, -0.08) > 0


def test_goodman_positive():
    assert goodman_correction(500, 50, 1000) > 500


def test_gerber_positive():
    assert gerber_correction(500, 50, 1000) > 500


def test_vm_positive():
    assert von_mises_stress(100, 50, 20, 10, 5, 3) > 0
'''

files["tests/test_fea.py"] = r'''
import pandas as pd
from pyfatiguepro.fea import FEAIngestor


def test_fea_ingestion():
    df = pd.DataFrame({
        "S11": [500], "S22": [100], "S33": [50],
        "S12": [10], "S23": [5], "S13": [3],
    })
    ing = FEAIngestor()
    out = ing.add_von_mises(df)
    assert "von_mises" in out.columns
    assert out["von_mises"].iloc[0] > 0
'''

files["tests/test_validation.py"] = r'''
import pandas as pd
from pyfatiguepro.validation import validation_report, ProprietaryDataValidator, DataProvenanceRegistry


def test_validation_report():
    r = validation_report([1000, 2000, 3000], [1100, 1900, 3100])
    assert r["MAPE_percent"] < 20


def test_prop_validator():
    df = pd.DataFrame({"stress_amplitude": [500], "cycles": [10000], "project": ["secret"]})
    v = ProprietaryDataValidator()
    assert v.validate_schema(df)
    anon = v.anonymize(df)
    assert anon["project"].iloc[0] != "secret"


def test_registry():
    reg = DataProvenanceRegistry()
    reg.add("x", "Ti64", "demo_synthetic", "generated", True)
    assert len(reg.to_frame()) == 1
'''

files["tests/test_ml.py"] = r'''
import numpy as np
import pandas as pd
from pyfatiguepro.validation import ProprietaryDataValidator
from pyfatiguepro.ml import FatigueMLPredictor


def test_ml_train_predict():
    n = 200
    rng = np.random.default_rng(1)
    stress = rng.uniform(300, 700, n)
    cycles = 1e7 * (stress / 300) ** -4
    df = pd.DataFrame({
        "material": ["Ti64"] * n,
        "stress_amplitude": stress,
        "temperature": rng.uniform(25, 500, n),
        "cycles": cycles,
    })
    v = ProprietaryDataValidator()
    df = v.clean_for_training(df)
    model = FatigueMLPredictor("rf")
    metrics = model.train(df, target_col="log_life")
    pred = model.predict({"material": "Ti64", "stress_amplitude": 500, "temperature": 100})
    assert pred[0] > 0
    assert "r2_log" in metrics
'''

files["Dockerfile"] = r'''
FROM python:3.11-slim
WORKDIR /app
COPY . /app
RUN pip install --upgrade pip && pip install -e ".[explain]"
EXPOSE 8002
CMD ["uvicorn", "pyfatiguepro.api:app", "--host", "0.0.0.0", "--port", "8002"]
'''

files[".github/workflows/ci.yml"] = r'''
name: CI
on:
  push:
    branches: ["main"]
  pull_request:
    branches: ["main"]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.9", "3.10", "3.11"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - name: Install
        run: |
          python -m pip install --upgrade pip
          pip install -e ".[dev]"
      - name: Lint
        run: ruff check pyfatiguepro tests
      - name: Test
        run: pytest --cov=pyfatiguepro --cov-report=term-missing
'''

files["LICENSE"] = r'''
MIT License

Copyright (c) 2026 RAVINDRANADH BOBBILI

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
'''


def write_files():
    for name, content in files.items():
        path = ROOT / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print(f"Created complete library at: {ROOT.resolve()}")
    print("\nRun these commands:")
    print("cd pyfatiguepro")
    print("pip install -e \".[dev,explain]\"")
    print("pytest")
    print("python examples/basquin_demo.py")
    print("python examples/fea_ingest_demo.py")
    print("python examples/train_demo.py")
    print("uvicorn pyfatiguepro.api:app --host 0.0.0.0 --port 8002")
    print("python -m build")
    print("twine check dist/*")
    print("twine upload --repository testpypi dist/*")
    print("twine upload dist/*")


if __name__ == "__main__":
    write_files()


Created complete library at: C:\Users\GOWREESWARI\pyfatiguepro

Run these commands:
cd pyfatiguepro
pip install -e ".[dev,explain]"
pytest
python examples/basquin_demo.py
python examples/fea_ingest_demo.py
python examples/train_demo.py
uvicorn pyfatiguepro.api:app --host 0.0.0.0 --port 8002
python -m build
twine check dist/*
twine upload --repository testpypi dist/*
twine upload dist/*


In [1]:
from pyfatiguepro.core import fit_basquin

stress = [760,700,650,600,550]
cycles = [1e4,3e4,8e4,2e5,7e5]

fit = fit_basquin(stress, cycles)
print(fit)

{'sigma_f_prime': 1629.6639968798875, 'b': -0.07695108823125127, 'predicted_cycles': array([ 10094.86529506,  29392.30833615,  76998.50176355, 217883.90752945,
       674990.44374417])}


In [3]:
import pyfatiguepro



In [4]:
from pyfatiguepro.core import goodman_correction, gerber_correction

stress_amp = 500
mean_stress = 100
ultimate_strength = 1000

goodman = goodman_correction(stress_amp, mean_stress, ultimate_strength)
gerber = gerber_correction(stress_amp, mean_stress, ultimate_strength)

print("Goodman corrected stress:", goodman)
print("Gerber corrected stress:", gerber)

Goodman corrected stress: 555.5555555555555
Gerber corrected stress: 505.0505050505051
